# RAG Assignment: Baseline Bi-Encoder Retrieval vs. Cross-Encoder Re-Ranking

**GitHub repository:** https://github.com/Sankalp0109/RAG_ASSIGNMENT_LMA

**Corpus:** *2024 FIA Formula One Sporting Regulations* (Issue 6, published 30 April 2024, 106 pages): `corpus_document.pdf`.
It is a dense regulatory document full of general rules followed by "unless…", "except…", "may not…" clauses,
cross-references between articles, and repeated vocabulary (*safety car, pit lane, penalty, tyres*) across many
unrelated articles, which makes it a good stress test for dense retrieval.

| Component | Choice |
|---|---|
| Chunking | Article-aware: one chunk per sub-article (e.g. `55.8`), long ones split at ≤900 chars with 150-char overlap |
| Bi-Encoder (dense retrieval) | `sentence-transformers/all-MiniLM-L6-v2` + FAISS `IndexFlatIP` on L2-normalised vectors (= cosine similarity) |
| Cross-Encoder (re-ranker) | `cross-encoder/ms-marco-MiniLM-L-6-v2` |
| LLM | Google Gemini (`gemini-3.6-flash`, temperature 0), same prompt for both pipelines |

**Pipelines**
* **Pipeline 1 (Baseline):** query → bi-encoder → FAISS Top-10 → take Top-3 → LLM → *Answer 1*
* **Pipeline 2 (Re-Ranking):** the same Top-10 → cross-encoder scores all 10 (query, chunk) pairs → re-sort → Top-3 → LLM (identical prompt) → *Answer 2*

**Notebook outline**
1. Setup & configuration
2. Load and chunk the PDF
3. Bi-encoder indexing (vector store)
4. Retrieval pipelines (baseline + re-ranking)
5. LLM answer generation
6. Failure-mode test suite (rank analysis on all queries)
7. Deep dive: rank shifts + Answer 1 vs Answer 2 for each failure query
8. Latency profiling
9. Scaling experiment: why not run the cross-encoder over the whole corpus?
10. Summary

## 1. Setup & configuration

In [1]:
# Install dependencies (uncomment on a fresh environment)
# %pip install torch sentence-transformers faiss-cpu pymupdf numpy pandas google-genai

import os, re, time, json, textwrap, statistics
from pathlib import Path
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import numpy as np
import pandas as pd
import torch
import faiss
import fitz  # PyMuPDF
from sentence_transformers import SentenceTransformer, CrossEncoder
from google import genai
from google.genai import types

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

# ---- configuration -------------------------------------------------------------
PDF_PATH        = "corpus_document.pdf"
BI_ENCODER      = "sentence-transformers/all-MiniLM-L6-v2"
CROSS_ENCODER   = "cross-encoder/ms-marco-MiniLM-L-6-v2"
LLM_MODEL       = "gemini-3.6-flash"
TOP_K_RETRIEVE  = 10   # candidates fetched from the vector store
TOP_K_CONTEXT   = 3    # chunks passed to the LLM
DEVICE = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")

# ---- API key: read from a local .env file (kept out of git) or the environment ----
def load_env(path=".env"):
    if Path(path).exists():
        for line in Path(path).read_text().splitlines():
            if "=" in line and not line.strip().startswith("#"):
                k, v = line.split("=", 1)
                os.environ.setdefault(k.strip(), v.strip().strip('"').strip("'"))
load_env()
assert os.environ.get("GOOGLE_API_KEY") or os.environ.get("GEMINI_API_KEY"), "Set GOOGLE_API_KEY in .env"

torch.manual_seed(0); np.random.seed(0)
print("torch", torch.__version__, "| device:", DEVICE)

torch 2.7.1 | device: mps


## 2. Load and chunk the PDF

The regulations are structured as *Article → sub-article → lettered clauses* (e.g. **55.8 a)–h)**).
Fixed-size chunking would cut rules in the middle of an exception list, so the chunker:

1. extracts text with PyMuPDF and strips the repeated page header/footer;
2. detects article headings (`36)` + `REFUELLING`) and sub-article numbers (`36.1`), ignoring cross-references such as
   "see Article 54.3" by requiring the number to belong to the current article and to increase monotonically;
3. creates one unit per sub-article, merges tiny neighbouring units of the same article, and splits long units
   (≤ 900 characters, about the 256-token window of MiniLM) on sentence/line boundaries with a 150-character overlap;
4. keeps metadata (article number/title, sub-article ids, pages) so we can check *which rule* each chunk holds.

Both pipelines use exactly the same chunks.

In [2]:
HEADER_FOOTER = [
    re.compile(r"^\s*2024 Formula 1 Sporting Regulations\s+\d+/\d+\s+30 April 2024\s*$"),
    re.compile(r"^\s*©2024 Fédération Internationale de l.Automobile\s+Issue 6\s*$"),
    re.compile(r"^\s*2024 Formula 1 Sporting Regulations\s*$"),
    re.compile(r"^\s*\d+/106\s*$"),
    re.compile(r"^\s*30 April 2024\s*$"),
    re.compile(r"^\s*©2024 Fédération Internationale de l.Automobile\s*$"),
    re.compile(r"^\s*Issue 6\s*$"),
]
ARTICLE_RE = re.compile(r"^(\d{1,2})\)$")          # PyMuPDF puts "36)" on its own line, title on the next
SUBART_RE = re.compile(r"^(\d{1,2})\.(\d{1,2})(?:\s+(.*))?$")  # "36.1" alone, or "26.13 Unless authorised ..."
APPENDIX_RE = re.compile(r"^APPENDIX \d+")
MARKER_RE = re.compile(r"^(\d{1,2}\.\d{1,2}|([a-z] )?[a-z]{1,2}\)|[ivx]+\))$")


def load_pages(pdf_path):
    """Return a list of (page_number, cleaned_text) with headers/footers removed."""
    doc = fitz.open(pdf_path)
    pages = []
    for i, page in enumerate(doc):
        lines = []
        for line in page.get_text("text").splitlines():
            if any(p.match(line) for p in HEADER_FOOTER):
                continue
            lines.append(line.rstrip())
        pages.append((i + 1, "\n".join(lines)))
    return pages


def split_long(text, max_chars, overlap):
    """Split an over-long unit on sentence/line boundaries with a small overlap.
    Returns (start, end, piece) triples so each piece can be mapped back to its pages."""
    pieces, start = [], 0
    while start < len(text):
        end = min(start + max_chars, len(text))
        if end < len(text):
            cut = max(text.rfind(". ", start, end), text.rfind("\n", start, end))
            if cut > start + max_chars // 2:
                end = cut + 1
        pieces.append((start, end, text[start:end].strip()))
        if end >= len(text):
            break
        # start the overlap on a word boundary so the next chunk never begins mid-word
        nxt = max(end - overlap, start + 1)
        space = text.find(" ", nxt, end)
        start = space + 1 if space != -1 else nxt
    return pieces


def chunk_document(pdf_path, max_chars=900, overlap=150, min_chars=200, skip_pages=1):
    """Article-aware chunking.

    1. Walk the text line by line, tracking the current top-level article ("36) REFUELLING")
       and the current sub-article number ("36.3").
    2. Each sub-article becomes one unit; tiny consecutive units of the same article are merged,
       over-long units are split with overlap.
    3. Every chunk keeps metadata: article number/title, sub-article ids and the pages it spans.
    """
    pages = load_pages(pdf_path)[skip_pages:]  # page 1 = table of contents
    lines = [(pg, l.strip()) for pg, text in pages for l in text.splitlines() if l.strip()]
    units, cur = [], None
    art_no, art_title, last_sub = None, None, 0
    i = 0
    while i < len(lines):
        page_no, s = lines[i]
        m_art, m_sub = ARTICLE_RE.match(s), SUBART_RE.match(s)
        # "54.3" only counts as a heading if it belongs to the current article and the
        # numbering moves forward (otherwise it is a cross-reference inside a sentence)
        if m_sub and (m_sub.group(1) != art_no or int(m_sub.group(2)) <= last_sub):
            m_sub = None
        if m_art and i + 1 < len(lines) and lines[i + 1][1].isupper():
            if cur:
                units.append(cur)
            art_no, art_title = m_art.group(1), lines[i + 1][1].title()
            cur, i, last_sub = None, i + 2, 0
            continue
        if APPENDIX_RE.match(s):
            if cur:
                units.append(cur)
            art_no, art_title = "App" + s.split()[1].rstrip(":"), s.title()
            cur = {"article": art_no, "title": art_title, "subs": [], "marks": [(0, page_no)], "text": s}
        elif m_sub:
            if cur:
                units.append(cur)
            sub_id = f"{m_sub.group(1)}.{m_sub.group(2)}"
            last_sub = int(m_sub.group(2))
            cur = {"article": art_no, "title": art_title, "subs": [sub_id], "marks": [(0, page_no)], "text": s}
        elif cur is not None:
            # glue list markers such as "a)" / "ii)" / "36.1" to the sentence that follows them
            last_line = cur["text"].rsplit("\n", 1)[-1]
            cur["text"] += (" " if MARKER_RE.match(last_line) else "\n")
            if page_no != cur["marks"][-1][1]:
                cur["marks"].append((len(cur["text"]), page_no))  # page break inside this unit
            cur["text"] += s
        i += 1
    if cur:
        units.append(cur)

    # merge tiny neighbours that belong to the same article
    merged = []
    for u in units:
        if merged and merged[-1]["article"] == u["article"] and \
                len(merged[-1]["text"]) < min_chars and len(merged[-1]["text"]) + len(u["text"]) <= max_chars:
            offset = len(merged[-1]["text"]) + 1
            merged[-1]["text"] += "\n" + u["text"]
            merged[-1]["subs"] += u["subs"]
            merged[-1]["marks"] += [(o + offset, p) for o, p in u["marks"]]
        else:
            merged.append(dict(u, subs=list(u["subs"]), marks=list(u["marks"])))

    chunks = []
    for u in merged:
        for start, end, part in split_long(u["text"], max_chars, overlap):
            # pages = page active at `start` plus any page that begins inside [start, end)
            pages = [p for o, p in u["marks"] if o <= start][-1:] + [p for o, p in u["marks"] if start < o < end]
            chunks.append({
                "chunk_id": len(chunks),
                "article": u["article"],
                "title": u["title"],
                "subs": u["subs"],
                "pages": sorted(set(pages)),
                "text": part,
            })
    return chunks

In [3]:
chunks = chunk_document(PDF_PATH)
texts = [c["text"] for c in chunks]
lens = [len(t) for t in texts]
print(f"{len(chunks)} chunks | {len({s for c in chunks for s in c['subs']})} sub-articles | "
      f"chars per chunk: min={min(lens)}, median={int(np.median(lens))}, max={max(lens)}")

def label(c):
    # Short label for a chunk, e.g. "55.8 (Safety Car)"
    sub = c["subs"][0] if c["subs"] else c["article"]
    return f"{sub} ({c['title'][:28]})"

pd.DataFrame([{"chunk_id": c["chunk_id"], "label": label(c), "pages": c["pages"], "text": c["text"][:110] + "…"}
              for c in chunks[::60]])

526 chunks | 324 sub-articles | chars per chunk: min=57, median=796, max=899


,chunk_id,label,pages,text
0,0,1.1 (Regulations),[2],1.1 The FIA will organise the FIA Formula One World Championship (the Championship) which is the\nproperty of t…
1,60,10.8 (Track Running Time Outside A),[13],the start of a ten (10) day period which precedes the\nstart of the second Competition and the last Competition…
2,120,24.1 (Competitor Factory Shutdown ),[23],24.1 All Competitors must observe two shutdown periods:\na) The first period of fourteen (14) consecutiv…
3,180,30.5 (Supply Of Tyres In The Champ),[35],of Article 32.4c)\nin which case two (2) sets must be returned.\niv) One (1) set of medium specification tyres m…
4,240,39.2 (Qualifying And Sprint Qualif),[45],39.2 The qualifying session will be run as follows:\na) For the first eighteen (18) minutes of the session (Q1)…
5,300,44.11 (Race Starting Procedure),[55],44.11 If a car is immobilised on the starting grid after the formation lap has commenced:\na) the marshals alon…
6,360,57.3 (Suspending A Sprint Session ),[64],57.3 Any cars unable to return to the pit lane as a result of the track being blocked will be brought\nback whe…
7,420,App6 (Appendix 6),[79],"Unit Manufacturers of an homologated power unit must:\n1.\nprovide the FIA, before 15 May (or such other date as…"
8,480,App7 (Appendix 7),[93],"considered as Restricted Aerodynamic testing, provided that\nthere is no direct or indirect measurement of aero…"


## 3. Bi-encoder indexing (vector store)

Every chunk is embedded **once, offline** by the bi-encoder. Vectors are L2-normalised so that the inner product in a
FAISS `IndexFlatIP` equals cosine similarity. (The corpus is small, so exact search is used; for millions of chunks an
ANN index such as HNSW/IVF would replace it.)

In [4]:
bi_encoder = SentenceTransformer(BI_ENCODER, device=DEVICE)
cross_encoder = CrossEncoder(CROSS_ENCODER, device=DEVICE)

t0 = time.perf_counter()
chunk_emb = bi_encoder.encode(texts, batch_size=64, normalize_embeddings=True,
                              convert_to_numpy=True, show_progress_bar=False).astype("float32")
index = faiss.IndexFlatIP(chunk_emb.shape[1])
index.add(chunk_emb)
index_time = time.perf_counter() - t0
print(f"Indexed {index.ntotal} chunks, dim={chunk_emb.shape[1]}, in {index_time:.2f}s (one-off cost)")

# warm-up both models so the first timed call does not include lazy initialisation
_ = bi_encoder.encode(["warm up"], normalize_embeddings=True)
_ = cross_encoder.predict([("warm up", texts[0])])

Indexed 526 chunks, dim=384, in 2.04s (one-off cost)


## 4. Retrieval pipelines

* `baseline_retrieve` = **Pipeline 1, steps a–b**: embed the query, take the Top-10 chunks by cosine similarity.
* `rerank` = **Pipeline 2, steps a–c**: score each of the *same* 10 (query, chunk) pairs jointly with the cross-encoder
  and re-sort by that score.

Each function also returns its run time so we can profile latency later.

In [5]:
def sync():
    # MPS/CUDA kernels run asynchronously; synchronise so the timings are honest
    if DEVICE == "mps": torch.mps.synchronize()
    elif DEVICE == "cuda": torch.cuda.synchronize()

def baseline_retrieve(query, k=TOP_K_RETRIEVE):
    # Pipeline 1: bi-encoder + vector search
    t0 = time.perf_counter()
    q = bi_encoder.encode([query], normalize_embeddings=True, convert_to_numpy=True).astype("float32")
    sync(); t1 = time.perf_counter()
    scores, ids = index.search(q, k)
    t2 = time.perf_counter()
    cands = [{"chunk_id": int(i), "bi_rank": r + 1, "bi_score": float(s)}
             for r, (i, s) in enumerate(zip(ids[0], scores[0]))]
    return cands, {"embed_ms": (t1 - t0) * 1e3, "search_ms": (t2 - t1) * 1e3}

def rerank(query, cands):
    # Pipeline 2: cross-encoder over the same Top-10 candidates
    t0 = time.perf_counter()
    ce_scores = cross_encoder.predict([(query, texts[c["chunk_id"]]) for c in cands], batch_size=len(cands))
    sync(); t1 = time.perf_counter()
    out = [dict(c, ce_score=float(s)) for c, s in zip(cands, ce_scores)]
    out.sort(key=lambda c: c["ce_score"], reverse=True)
    for r, c in enumerate(out):
        c["ce_rank"] = r + 1
    return out, {"rerank_ms": (t1 - t0) * 1e3}

# quick sanity check
cands, t = baseline_retrieve("What is the pit lane speed limit?")
reranked, t2 = rerank("What is the pit lane speed limit?", cands)
print([label(chunks[c["chunk_id"]]) for c in cands[:3]], t)
print([label(chunks[c["chunk_id"]]) for c in reranked[:3]], t2)

['34.6 (Pit Entry Road, Pit Lane And)', '34.3 (Pit Entry Road, Pit Lane And)', '57.2 (Suspending A Sprint Session )'] {'embed_ms': 30.647457999293692, 'search_ms': 1.5385420010716189}
['34.6 (Pit Entry Road, Pit Lane And)', '44.7 (Race Starting Procedure)', '34.3 (Pit Entry Road, Pit Lane And)'] {'rerank_ms': 67.13945800038346}


## 5. LLM answer generation

Both pipelines use the **identical prompt**; the only difference is which three chunks fill the context. The model is told
to answer strictly from the context and to cite article numbers, so a missing chunk shows up as an omitted clause, a
wrong rule, or an admission that the context is insufficient, rather than the model quietly using what it already
knows about F1.

In [6]:
client = genai.Client(api_key=os.environ.get("GOOGLE_API_KEY") or os.environ.get("GEMINI_API_KEY"))

PROMPT_TEMPLATE = '''You are an assistant answering questions about the 2024 FIA Formula One Sporting Regulations.
Answer the question using ONLY the context passages below. Do not use outside knowledge.
Cite the article numbers you rely on. If the passages do not contain the information needed,
say "The provided context does not answer this." Keep the answer to at most 4 sentences.

Context:
{context}

Question: {question}
Answer:'''

def build_context(cands):
    return "\n\n".join(f"[Passage {i + 1} | Article {label(chunks[c['chunk_id']])}]\n{texts[c['chunk_id']]}"
                       for i, c in enumerate(cands[:TOP_K_CONTEXT]))

# The free Gemini tier allows only ~20 requests/day per model, so every response is cached on disk together with
# the latency measured when it was generated. Re-running the notebook re-uses these answers (and their timings)
# instead of spending quota; delete results/llm_cache.json to force fresh calls.
import hashlib
CACHE_PATH = Path("results/llm_cache.json")
llm_cache = json.loads(CACHE_PATH.read_text()) if CACHE_PATH.exists() else {}

def generate_answer(question, cands, retries=4):
    prompt = PROMPT_TEMPLATE.format(context=build_context(cands), question=question)
    key = hashlib.sha256(f"{LLM_MODEL}\n{prompt}".encode()).hexdigest()
    if key in llm_cache:
        return llm_cache[key]["text"], llm_cache[key]["latency_ms"]
    for attempt in range(retries):
        try:
            t0 = time.perf_counter()
            resp = client.models.generate_content(
                model=LLM_MODEL, contents=prompt,
                config=types.GenerateContentConfig(temperature=0.0))
            text, ms = resp.text.strip(), (time.perf_counter() - t0) * 1e3
            llm_cache[key] = {"text": text, "latency_ms": ms, "question": question}
            CACHE_PATH.parent.mkdir(exist_ok=True)
            CACHE_PATH.write_text(json.dumps(llm_cache, indent=1))
            return text, ms
        except Exception as e:  # rate limits / transient network errors
            wait = 5 * (attempt + 1)
            print(f"LLM error ({type(e).__name__}: {str(e)[:80]}), retrying in {wait}s")
            time.sleep(wait)
    return "[LLM call failed]", float("nan")

print(generate_answer("What is the pit lane speed limit?", reranked)[0])

According to Article 34.7, the pit lane speed limit is 80 km/h during the whole Competition. However, this limit may be amended by the Race Director following a recommendation from the Safety Delegate.


## 6. Failure-mode test suite

Each query has a **gold sub-article** that holds the correct answer (checked by hand against the PDF) and is tagged with
the failure mode it is designed to probe:

* **Keyword trap / distractor**: the query shares words with an irrelevant passage, while the answer is phrased differently.
* **Exception / corner case**: a general rule exists, but the answer depends on an "unless / except" clause.
* **Negation / constraint**: "not allowed", "prohibited", "may not" questions.
* **Buried chunk**: the vital chunk exists in the Top-10 but sits at rank 4–10.

For every query we record the gold chunk's rank under the bi-encoder (Pipeline 1) and after cross-encoder re-ranking
(Pipeline 2). Because one sub-article can span several chunks, the gold is pinned to the *exact sentence* that answers
the question (`gold_text`): a chunk is gold if it contains that sentence (whitespace-normalised).

In [7]:
QUERIES = [{'id': 'Q01',
  'category': 'Keyword trap / distractor',
  'query': 'Can a car be refuelled during a pit stop in the race?',
  'gold': ['36.2'],
  'gold_text': ['Fuel may not be added to nor removed from a car after it has left the pit lane']},
 {'id': 'Q02',
  'category': 'Negation / constraint',
  'query': 'Is refuelling allowed during the race?',
  'gold': ['36.2'],
  'gold_text': ['Fuel may not be added to nor removed from a car after it has left the pit lane']},
 {'id': 'Q03',
  'category': 'Keyword trap / distractor',
  'query': 'Can the team top up fuel when the car comes in for new tyres?',
  'gold': ['36.2'],
  'gold_text': ['Fuel may not be added to nor removed from a car after it has left the pit lane'],
  'truth': 'No. Fuel may not be added to or removed from a car after it has left the pit lane for the reconnaissance '
           "lap(s) until the end-of-session signal; refuelling is only allowed in the team's designated garage.",
  'deep_dive': True,
  'regression': True,
  'observation': 'The baseline had the refuelling chunk (36.1/36.2) at #3, and **Answer 1 is correct** (fuel may '
                 'only be added in the garage and not between the reconnaissance lap and the end of the race). The '
                 'query is phrased around *tyres*, so the cross-encoder fills the Top-3 with tyre and test-day '
                 'chunks. Every cross-encoder score here is strongly negative (−5.6 to −10.7), meaning it finds no '
                 'candidate convincing, and it pushes the refuelling chunk to #5. **Answer 2** says the context does '
                 'not answer the question. MS MARCO re-rankers are trained on short web queries and can over-weight '
                 'the dominant topic word in a mixed-topic question.'},
 {'id': 'Q04',
  'category': 'Control (direct lookup)',
  'query': 'What is the maximum speed allowed in the pit lane?',
  'gold': ['34.7'],
  'gold_text': ['A speed limit of 80km/h']},
 {'id': 'Q05',
  'category': 'Control (direct lookup)',
  'query': 'How many points does a driver get for setting the fastest lap?',
  'gold': ['6.4'],
  'gold_text': ['one (1) point will be awarded to the driver who achieved the fastest']},
 {'id': 'Q06',
  'category': 'Exception / corner case',
  'query': 'Is the driver of the fastest lap guaranteed a point?',
  'gold': ['6.4'],
  'gold_text': ['one (1) point will be awarded to the driver who achieved the fastest']},
 {'id': 'Q07',
  'category': 'Exception / corner case',
  'query': 'Does a driver who sets the fastest lap but finishes 12th get a bonus point?',
  'gold': ['6.4'],
  'gold_text': ['one (1) point will be awarded to the driver who achieved the fastest']},
 {'id': 'Q08',
  'category': 'Control (direct lookup)',
  'query': 'What happens if a driver collects too many penalty points?',
  'gold': ['4.2'],
  'gold_text': ['If a driver accrues twelve (12) penalty points']},
 {'id': 'Q09',
  'category': 'Keyword trap / distractor',
  'query': 'When does a driver receive a race ban?',
  'gold': ['4.2 / 54.3i'],
  'gold_text': ['If a driver accrues twelve (12) penalty points', 'Suspension from the driver']},
 {'id': 'Q10',
  'category': 'Exception / corner case',
  'query': 'Must every driver use two different tyre compounds in the race?',
  'gold': ['30.5m'],
  'gold_text': ['use at least two different specifications of dry-weather tyres'],
  'truth': 'Yes, unless the driver used intermediate or wet-weather tyres during the race: each driver must use at '
           'least two different dry-weather specifications, at least one of them a mandatory race specification. '
           'Failure = disqualification (or +30 s if the race is suspended and cannot be restarted).',
  'deep_dive': True,
  'observation': 'The chunk holding the rule sat at bi-encoder rank #8, behind seven other tyre chunks that share '
                 'the words *tyre / specification / race* (30.3 tyre control, 30.5 allocation). The cross-encoder '
                 'reads the query and the passage together, sees that only this one states *each driver must use at '
                 'least two different specifications*, and moves it to #1. **Answer 1** says the context does not '
                 'answer the question; **Answer 2** states the rule and the disqualification penalty. Answer 2 still '
                 '**omits the "unless he has used intermediate or wet-weather tyres" exception**: the chunk boundary '
                 'falls inside clause 30.5 m), so the opening words of the exception are in the previous chunk. '
                 'Re-ranking cannot fix a chunking error.'},
 {'id': 'Q11',
  'category': 'Exception / corner case',
  'query': 'Is a driver who uses intermediate tyres in the race still required to use two dry compounds?',
  'gold': ['30.5m'],
  'gold_text': ['use at least two different specifications of dry-weather tyres']},
 {'id': 'Q12',
  'category': 'Buried chunk (rank 4-10)',
  'query': 'What is the penalty for not using two different dry tyre specifications?',
  'gold': ['30.5m'],
  'gold_text': ['failure to comply with this requirement will result in the disqualification'],
  'truth': 'Disqualification from the race results; if the race is suspended and cannot be re-started, 30 seconds '
           "are added to the driver's elapsed time instead.",
  'deep_dive': True,
  'observation': 'The gold chunk is buried at #9 and the baseline Top-3 contains a *different* tyre penalty (54.3d '
                 'for not changing tyres as specified). **Answer 1 misstates the rule**: it gives that unrelated '
                 '54.3d penalty as the penalty for the two-compound rule. After re-ranking (#9 → #1), **Answer 2** '
                 'gives the correct sanction: disqualification, or +30 s if the race is suspended and cannot be '
                 're-started.'},
 {'id': 'Q13',
  'category': 'Negation / constraint',
  'query': 'Can drivers overtake while the safety car is deployed?',
  'gold': ['55.8'],
  'gold_text': ['no driver may overtake another car on the track, including the safety car']},
 {'id': 'Q14',
  'category': 'Negation / constraint',
  'query': 'Can a driver overtake under the virtual safety car?',
  'gold': ['56.6'],
  'gold_text': ['no driver may overtake another car on the track whilst the VSC']},
 {'id': 'Q15',
  'category': 'Negation / constraint',
  'query': 'Can a driver enter the pits during a VSC?',
  'gold': ['56.4'],
  'gold_text': ['no car may enter the pits whilst the VSC procedure is in use']},
 {'id': 'Q16',
  'category': 'Negation / constraint',
  'query': 'Can a driver pit while the safety car is out?',
  'gold': ['55.11'],
  'gold_text': ['no car may enter the pits whilst the safety car is deployed']},
 {'id': 'Q17',
  'category': 'Exception / corner case',
  'query': 'How long is the race at Monaco?',
  'gold': ['5.4a'],
  'gold_text': ['The distance of the race in Monaco']},
 {'id': 'Q18',
  'category': 'Control (direct lookup)',
  'query': 'What is the minimum race distance in kilometres?',
  'gold': ['5.4'],
  'gold_text': ['exceed a distance of 305km']},
 {'id': 'Q19',
  'category': 'Exception / corner case',
  'query': 'Can a driver serve a drive-through penalty while the safety car is deployed?',
  'gold': ['54.4b'],
  'gold_text': ['he may not carry out the penalty if the VSC procedure is in use']},
 {'id': 'Q20',
  'category': 'Exception / corner case',
  'query': 'Can a driver serve a five second time penalty during a safety car pit stop?',
  'gold': ['54.3a-b'],
  'gold_text': ['this includes any stop the driver makes whilst a VSC or safety car']},
 {'id': 'Q21',
  'category': 'Negation / constraint',
  'query': 'Can mechanics work on the car while a driver serves a five second penalty?',
  'gold': ['54.4c'],
  'gold_text': ['it may not be worked on until the car has been stationary for the duration'],
  'truth': 'No. While stationary for a 5 s or 10 s time penalty the car may not be worked on until it has been '
           'stationary for the full penalty; touching the car or driver by hand, tools or equipment counts as work. '
           'A breach may lead to disqualification.',
  'deep_dive': True,
  'observation': 'A negation query: the answer is the clause *may not be worked on until … stationary for the '
                 'duration of the penalty* (54.4c), found at baseline rank #4. The baseline Top-3 holds 54.4d (the '
                 'stop-and-go clause) plus two irrelevant restart-procedure chunks. **Answer 1** reaches the right '
                 'yes/no answer but cites the wrong clause and adds the engine-restart allowance, which applies to '
                 'the stop-and-go penalty, not to a 5 s penalty. **Answer 2** cites 54.4c and the definition of '
                 '"working".'},
 {'id': 'Q22',
  'category': 'Negation / constraint',
  'query': 'What are team personnel prohibited from doing during the curfew?',
  'gold': ['23.5'],
  'gold_text': ['are not permitted within the confines of the circuit during the restricted periods',
                'must not conduct operational activities']},
 {'id': 'Q23',
  'category': 'Keyword trap / distractor',
  'query': 'Can engineers work on their computers at the track during the overnight restricted period?',
  'gold': ['23.5d'],
  'gold_text': ['must not conduct operational activities']},
 {'id': 'Q24',
  'category': 'Keyword trap / distractor',
  'query': 'How many times can a team break the overnight curfew in a season?',
  'gold': ['23.6'],
  'gold_text': ['Four (4) exceptions in each Championship']},
 {'id': 'Q25',
  'category': 'Negation / constraint',
  'query': 'Who is not allowed in the pit lane?',
  'gold': ['26.13'],
  'gold_text': ['no one under the age of 16']},
 {'id': 'Q26',
  'category': 'Keyword trap / distractor',
  'query': 'Can children enter the pit lane?',
  'gold': ['26.13'],
  'gold_text': ['no one under the age of 16'],
  'truth': 'Not at certain times: unless authorised by the FIA, no one under 16 may be in the pit lane from 15 min '
           'before to 5 min after each practice / qualifying / sprint-qualifying session, and from 15 min before the '
           'pit exit opens for reconnaissance laps until the last car enters parc fermé after the sprint or race.',
  'deep_dive': True,
  'observation': 'A keyword trap: the query says *children*, the regulation says *no one under the age of 16*. The '
                 'bi-encoder ranks generic pit-lane chunks ("car reversed in the pit lane", "pit exit will be '
                 'closed") above the answer because they share *pit lane*. The cross-encoder links *children* with '
                 '*under the age of 16* (#5 → #2). **Answer 1** says the context does not answer the question; '
                 '**Answer 2** gives the age limit, the restricted time windows and the FIA-authorisation '
                 'exception.'},
 {'id': 'Q27',
  'category': 'Keyword trap / distractor',
  'query': 'Are dogs allowed in the paddock?',
  'gold': ['26.16'],
  'gold_text': ['Animals, except those']},
 {'id': 'Q28',
  'category': 'Negation / constraint',
  'query': 'Are tyre blankets permitted in the pit lane?',
  'gold': ['26.17'],
  'gold_text': ['tyre blankets are not permitted in the Pit Lane']},
 {'id': 'Q29',
  'category': 'Negation / constraint',
  'query': 'Is a car allowed to reverse in the pit lane?',
  'gold': ['34.3'],
  'gold_text': ['At no time may a car be reversed in the pit lane']},
 {'id': 'Q30',
  'category': 'Negation / constraint',
  'query': 'What changes are teams not allowed to make to the car under parc ferme?',
  'gold': ['40.9'],
  'gold_text': ['may not modify any part on the car']},
 {'id': 'Q31',
  'category': 'Negation / constraint',
  'query': 'When are no championship points awarded for a shortened race?',
  'gold': ['6.5a-b'],
  'gold_text': ['No points will be awarded If the leader has completed less than two'],
  'truth': 'No points if the leader completed fewer than two laps; in the reduced-points cases, no points unless the '
           'leader completed at least two laps without Safety Car / VSC intervention.',
  'deep_dive': True,
  'observation': 'The question asks when **no** points are awarded. The bi-encoder ranks the *positive* points table '
                 '(6.4) and the dead-heat rule (7.1) first, and the second half of 6.5 at #3, but that half only has '
                 'the 25 %/50 %/75 % thresholds. The chunk with *No points will be awarded if the leader has '
                 'completed less than two laps* moves from #4 to #2. **Answer 1** says the context does not answer '
                 'the question; **Answer 2** gives both no-points conditions (fewer than 2 laps; fewer than 2 laps '
                 'without SC/VSC).'},
 {'id': 'Q32',
  'category': 'Exception / corner case',
  'query': 'How many points are given if the race is stopped early after only a few laps?',
  'gold': ['6.5'],
  'gold_text': ['No points will be awarded If the leader has completed less than two',
                'If the leader has completed two (2) laps but less than 25%']},
 {'id': 'Q33',
  'category': 'Exception / corner case',
  'query': 'What happens if two drivers set an identical lap time in qualifying?',
  'gold': ['39.4a)iv'],
  'gold_text': ['set identical times during Q1']},
 {'id': 'Q34',
  'category': 'Keyword trap / distractor',
  'query': 'Who wins the championship if two drivers finish level on points?',
  'gold': ['7.2'],
  'gold_text': ['The holder of the greatest number of first places']},
 {'id': 'Q35',
  'category': 'Control (direct lookup)',
  'query': 'How many engines can a driver use in a season?',
  'gold': ['28.2'],
  'gold_text': ['may use no more than four (4) engines']},
 {'id': 'Q36',
  'category': 'Exception / corner case',
  'query': 'Can a mechanic use an external starter to restart the engine on the grid during the race?',
  'gold': ['26.8'],
  'gold_text': ['the engine may only be started with the starter except']},
 {'id': 'Q37',
  'category': 'Negation / constraint',
  'query': 'Can a driver leave the track to gain an advantage?',
  'gold': ['33.3'],
  'gold_text': ['without gaining any lasting advantage']},
 {'id': 'Q38',
  'category': 'Control (direct lookup)',
  'query': 'What is the maximum number of team personnel allowed to work on the cars?',
  'gold': ['23.1'],
  'gold_text': ['fifty-eight (58) team personnel']},
 {'id': 'Q39',
  'category': 'Control (direct lookup)',
  'query': 'Can the race director close the pit entry?',
  'gold': ['34.15'],
  'gold_text': ['the Race Director may ask for the pit entry to be closed']},
 {'id': 'Q40',
  'category': 'Keyword trap / distractor',
  'query': 'What happens if a driver jumps the start?',
  'gold': ['48.1'],
  'gold_text': ['Moved after the four (4) second light is illuminated']},
 {'id': 'Q41',
  'category': 'Negation / constraint',
  'query': 'Can a driver appeal a time penalty given during the race?',
  'gold': ['54.3'],
  'gold_text': ['they shall not be subject to appeal', 'Appeals may not be made']},
 {'id': 'Q42',
  'category': 'Negation / constraint',
  'query': 'Which decisions cannot be appealed?',
  'gold': ['17.3 / 54.3'],
  'gold_text': ['they shall not be subject to appeal', 'Appeals may not be made']},
 {'id': 'Q43',
  'category': 'Control (direct lookup)',
  'query': 'How fast can fuel be pumped into the car?',
  'gold': ['36.3'],
  'gold_text': ['at a rate greater than 0.8 litres']},
 {'id': 'Q44',
  'category': 'Control (direct lookup)',
  'query': 'Must the engine be switched off while fuel is being added?',
  'gold': ['36.4'],
  'gold_text': ['but the engine must be stopped']},
 {'id': 'Q45',
  'category': 'Exception / corner case',
  'query': 'What happens to the two-compound tyre rule if the race is suspended and cannot be restarted?',
  'gold': ['30.5m'],
  'gold_text': ['thirty (30) seconds will be added to the elapsed time of any driver who was unable']},
 {'id': 'Q46',
  'category': 'Keyword trap / distractor',
  'query': 'Can a reserve driver take part in free practice?',
  'gold': ['32.4'],
  'gold_text': ['permitted to use additional drivers during P1 and P2']},
 {'id': 'Q47',
  'category': 'Buried chunk (rank 4-10)',
  'query': 'How many times per season must a team run a rookie driver in practice?',
  'gold': ['32.4c'],
  'gold_text': ['who has not participated in more than two (2)']},
 {'id': 'Q48',
  'category': 'Control (direct lookup)',
  'query': 'What is the penalty for an unsafe release in the race?',
  'gold': ['34.14d'],
  'gold_text': ['released in an unsafe condition during a sprint session or a race']},
 {'id': 'Q49',
  'category': 'Exception / corner case',
  'query': 'What happens if a penalty is given in the last three laps of the race?',
  'gold': ['54.3'],
  'gold_text': ['imposed during the last three (3) laps']},
 {'id': 'Q50',
  'category': 'Buried chunk (rank 4-10)',
  'query': 'What happens if a driver cannot serve a penalty because he retired?',
  'gold': ['54.3'],
  'gold_text': ['unable to serve the penalty due to retirement'],
  'truth': 'If a 5 s / 10 s / drive-through / stop-and-go penalty cannot be served because the driver retired, the '
           "stewards may impose a grid place penalty at the driver's next race.",
  'deep_dive': True,
  'observation': 'An exception clause buried at #6 in the middle of the long 54.3 penalty list. The bi-encoder '
                 'prefers chunks with *penalty / grid positions* (34.14, 49.5). The cross-encoder scores the clause '
                 '*unable to serve the penalty due to retirement* far higher (+5.54 vs at most −0.26 for every other '
                 'candidate) and moves it to #1. **Answer 1** says the context does not answer the question; '
                 '**Answer 2** gives the grid-place penalty at the next race.'},
 {'id': 'Q51',
  'category': 'Exception / corner case',
  'query': 'When are trainee mechanics allowed in addition to the operational personnel limit?',
  'gold': ['23.3'],
  'gold_text': ['sixteen (16) individual exceptions during a Championship for trainee personnel']},
 {'id': 'Q52',
  'category': 'Control (direct lookup)',
  'query': 'Do laps behind the VSC count towards race distance?',
  'gold': ['56.8'],
  'gold_text': ['counted as a lap']}]

In [8]:
def norm(t):
    return " ".join(t.split()).lower()

def is_gold(chunk_id, q):
    return any(norm(g) in norm(texts[chunk_id]) for g in q["gold_text"])

for q in QUERIES:  # every gold sentence must exist in the corpus
    assert any(is_gold(i, q) for i in range(len(chunks))), q["id"]

def best_rank(ranked, q, key):
    ranks = [c[key] for c in ranked if is_gold(c["chunk_id"], q)]
    return min(ranks) if ranks else None   # None = gold not in the Top-10 at all

results = {}
rows = []
for q in QUERIES:
    cands, t_b = baseline_retrieve(q["query"])
    reranked, t_r = rerank(q["query"], cands)
    results[q["id"]] = {"cands": cands, "reranked": reranked}
    b, r = best_rank(cands, q, "bi_rank"), best_rank(reranked, q, "ce_rank")
    rows.append({"id": q["id"], "category": q["category"], "query": q["query"], "gold": ", ".join(q["gold"]),
                 "baseline_rank": b, "reranked_rank": r,
                 "shift": f"#{b} → #{r}" if b else "not in Top-10",
                 "base_top3": bool(b and b <= 3), "rerank_top3": bool(r and r <= 3)})
rank_df = pd.DataFrame(rows)
rank_df["baseline_rank"] = rank_df["baseline_rank"].astype("Int64")
rank_df["reranked_rank"] = rank_df["reranked_rank"].astype("Int64")
rank_df

,id,category,query,gold,baseline_rank,reranked_rank,shift,base_top3,rerank_top3
0,Q01,Keyword trap / distractor,Can a car be refuelled during a pit stop in the race?,36.2,1,1,#1 → #1,True,True
1,Q02,Negation / constraint,Is refuelling allowed during the race?,36.2,1,1,#1 → #1,True,True
2,Q03,Keyword trap / distractor,Can the team top up fuel when the car comes in for new tyres?,36.2,3,5,#3 → #5,True,False
3,Q04,Control (direct lookup),What is the maximum speed allowed in the pit lane?,34.7,1,1,#1 → #1,True,True
4,Q05,Control (direct lookup),How many points does a driver get for setting the fastest lap?,6.4,2,1,#2 → #1,True,True
5,Q06,Exception / corner case,Is the driver of the fastest lap guaranteed a point?,6.4,2,1,#2 → #1,True,True
6,Q07,Exception / corner case,Does a driver who sets the fastest lap but finishes 12th get a bonus point?,6.4,1,1,#1 → #1,True,True
7,Q08,Control (direct lookup),What happens if a driver collects too many penalty points?,4.2,1,1,#1 → #1,True,True
8,Q09,Keyword trap / distractor,When does a driver receive a race ban?,4.2 / 54.3i,1,1,#1 → #1,True,True
9,Q10,Exception / corner case,Must every driver use two different tyre compounds in the race?,30.5m,8,1,#8 → #1,False,True


In [9]:
def mrr(col, df=None):
    df = rank_df if df is None else df
    return np.mean([0 if pd.isna(v) else 1 / v for v in df[col]])

summary = pd.DataFrame({
    "metric": ["Hit@3 (gold in context)", "MRR@10", "Hit@1"],
    "Pipeline 1: Baseline": [rank_df.base_top3.mean(), mrr("baseline_rank"), (rank_df.baseline_rank == 1).fillna(False).mean()],
    "Pipeline 2: Re-ranked": [rank_df.rerank_top3.mean(), mrr("reranked_rank"), (rank_df.reranked_rank == 1).fillna(False).mean()],
}).round(3)
in_top10 = rank_df.baseline_rank.notna()
print(f"{len(rank_df)} queries | gold inside Top-10 for {int(in_top10.sum())} "
      f"(re-ranking can only reorder these) | rescued into Top-3 by re-ranking: "
      f"{int(((~rank_df.base_top3) & rank_df.rerank_top3).sum())} | "
      f"pushed out of Top-3 by re-ranking: {int((rank_df.base_top3 & ~rank_df.rerank_top3).sum())}")
summary

52 queries | gold inside Top-10 for 47 (re-ranking can only reorder these) | rescued into Top-3 by re-ranking: 6 | pushed out of Top-3 by re-ranking: 2


,metric,Pipeline 1: Baseline,Pipeline 2: Re-ranked
0,Hit@3 (gold in context),0.769,0.846
1,MRR@10,0.656,0.774
2,Hit@1,0.519,0.692


In [10]:
# Breakdown per failure category
per_cat = rank_df.groupby("category").apply(lambda d: pd.Series({
    "queries": len(d),
    "Hit@3 baseline": d.base_top3.mean(), "Hit@3 re-ranked": d.rerank_top3.mean(),
    "MRR@10 baseline": mrr("baseline_rank", d), "MRR@10 re-ranked": mrr("reranked_rank", d),
}), include_groups=False).round(2)
per_cat

,queries,Hit@3 baseline,Hit@3 re-ranked,MRR@10 baseline,MRR@10 re-ranked
category,,,,,
Buried chunk (rank 4-10),3.0,0.33,0.67,0.20,0.73
Control (direct lookup),11.0,1.00,1.00,0.95,0.94
Exception / corner case,13.0,0.85,0.92,0.64,0.83
Keyword trap / distractor,10.0,0.60,0.60,0.50,0.57
Negation / constraint,15.0,0.73,0.87,0.65,0.75


In [11]:
# All queries whose gold chunk moved across the Top-3 boundary (the cases that change what the LLM sees)
moved = rank_df[rank_df.base_top3 != rank_df.rerank_top3][["id", "category", "query", "gold", "shift"]]
moved

,id,category,query,gold,shift
2,Q03,Keyword trap / distractor,Can the team top up fuel when the car comes in for new tyres?,36.2,#3 → #5
9,Q10,Exception / corner case,Must every driver use two different tyre compounds in the race?,30.5m,#8 → #1
11,Q12,Buried chunk (rank 4-10),What is the penalty for not using two different dry tyre specifications?,30.5m,#9 → #1
20,Q21,Negation / constraint,Can mechanics work on the car while a driver serves a five second penalty?,54.4c,#4 → #2
25,Q26,Keyword trap / distractor,Can children enter the pit lane?,26.13,#5 → #2
30,Q31,Negation / constraint,When are no championship points awarded for a shortened race?,6.5a-b,#4 → #2
46,Q47,Buried chunk (rank 4-10),How many times per season must a team run a rookie driver in practice?,32.4c,#3 → #5
49,Q50,Buried chunk (rank 4-10),What happens if a driver cannot serve a penalty because he retired?,54.3,#6 → #1


## 7. Deep dive: rank shifts and Answer 1 vs Answer 2

For each failure query we show the full Top-10 with both ranks (★ = gold chunk), then generate **Answer 1** (baseline Top-3)
and **Answer 2** (re-ranked Top-3) with the identical prompt.

In [12]:
llm_log = []

def show_query(qid):
    q = next(x for x in QUERIES if x["id"] == qid)
    res = results[qid]
    print("=" * 110)
    print(f"[{q['id']}] {q['category']}\nQ: {q['query']}\nGround truth ({', '.join(q['gold'])}): {q['truth']}")
    print("=" * 110)
    ce_by_id = {c["chunk_id"]: c for c in res["reranked"]}
    table = pd.DataFrame([{
        "gold": "★" if is_gold(c["chunk_id"], q) else "",
        "chunk": label(chunks[c["chunk_id"]]),
        "bi_rank": c["bi_rank"], "bi_score": round(c["bi_score"], 3),
        "ce_rank": ce_by_id[c["chunk_id"]]["ce_rank"], "ce_score": round(ce_by_id[c["chunk_id"]]["ce_score"], 2),
        "text": texts[c["chunk_id"]][:90].replace("\n", " ") + "…"} for c in res["cands"]])
    display(table)
    a1, t1 = generate_answer(q["query"], res["cands"])
    a2, t2 = generate_answer(q["query"], res["reranked"])
    llm_log.extend([t1, t2])
    print("ANSWER 1 (baseline Top-3:", [label(chunks[c["chunk_id"]]) for c in res["cands"][:3]], ")")
    print(textwrap.fill(a1, 110), "\n")
    print("ANSWER 2 (re-ranked Top-3:", [label(chunks[c["chunk_id"]]) for c in res["reranked"][:3]], ")")
    print(textwrap.fill(a2, 110))
    q["answer_1"], q["answer_2"] = a1, a2

### Q10: Exception / corner case
**Query:** *Must every driver use two different tyre compounds in the race?*

In [13]:
show_query("Q10")

[Q10] Exception / corner case
Q: Must every driver use two different tyre compounds in the race?
Ground truth (30.5m): Yes, unless the driver used intermediate or wet-weather tyres during the race: each driver must use at least two different dry-weather specifications, at least one of them a mandatory race specification. Failure = disqualification (or +30 s if the race is suspended and cannot be restarted).


,gold,chunk,bi_rank,bi_score,ce_rank,ce_score,text
0,,30.3 (Supply Of Tyres In The Champ),1,0.683,2,3.62,one must present both tyres to the Technical Delegate. e) The use of tyres without appropr…
1,,30.5 (Supply Of Tyres In The Champ),2,0.680,5,2.19,30.5 Use of Tyres a) All tyres must be operated in accordance with the prescriptions issue…
2,,30.5 (Supply Of Tyres In The Champ),3,0.653,4,2.26,a set of tyres of the same specification. A penalty under Article 54.3d) will be imposed o…
3,,30.3 (Supply Of Tyres In The Champ),4,0.642,6,0.32,30.3 Control of tyres a) The outer sidewall of all tyres which are to be used at a Competi…
4,,30.5 (Supply Of Tyres In The Champ),5,0.607,7,0.16,"by the FIA. Unless the additional tyres have not been made available, or the session is de…"
5,,30.6 (Supply Of Tyres In The Champ),6,0.582,10,-5.37,30.6 Testing of tyres a) From the time at which each Competitor receives fitted tyres from…
6,,30.2 (Supply Of Tyres In The Champ),7,0.581,9,-2.29,"to each Competition for item iv) below, the FIA will provide all Competitors with the foll…"
7,★,30.5 (Supply Of Tyres In The Champ),8,0.580,1,7.08,"or wet-weather tyres during the race, each driver must use at least two different specific…"
8,,30.5 (Supply Of Tyres In The Champ),9,0.575,8,-1.29,"session or race is resumed in accordance with Article 58.10a), the use of wet- weather tyr…"
9,,30.2 (Supply Of Tyres In The Champ),10,0.571,3,2.59,where a sprint session is not scheduled and additional tyres are made available under Arti…


ANSWER 1 (baseline Top-3: ['30.3 (Supply Of Tyres In The Champ)', '30.5 (Supply Of Tyres In The Champ)', '30.5 (Supply Of Tyres In The Champ)'] )
The provided context does not answer this. 

ANSWER 2 (re-ranked Top-3: ['30.5 (Supply Of Tyres In The Champ)', '30.3 (Supply Of Tyres In The Champ)', '30.2 (Supply Of Tyres In The Champ)'] )
According to Article 30.5, each driver must use at least two different specifications of dry-weather tyres
during the race, at least one of which must be a mandatory dry-weather race tyre specification. Failure to
comply with this requirement will result in the disqualification of the relevant driver from the race results.
If the race is suspended and cannot be re-started, thirty seconds will instead be added to the elapsed time of
any driver who was unable to use at least two specifications of dry-weather tyres (Article 30.5).


**Observation:** The chunk holding the rule sat at bi-encoder rank #8, behind seven other tyre chunks that share the words *tyre / specification / race* (30.3 tyre control, 30.5 allocation). The cross-encoder reads the query and the passage together, sees that only this one states *each driver must use at least two different specifications*, and moves it to #1. **Answer 1** says the context does not answer the question; **Answer 2** states the rule and the disqualification penalty. Answer 2 still **omits the "unless he has used intermediate or wet-weather tyres" exception**: the chunk boundary falls inside clause 30.5 m), so the opening words of the exception are in the previous chunk. Re-ranking cannot fix a chunking error.

### Q12: Buried chunk (rank 4-10)
**Query:** *What is the penalty for not using two different dry tyre specifications?*

In [14]:
show_query("Q12")

[Q12] Buried chunk (rank 4-10)
Q: What is the penalty for not using two different dry tyre specifications?
Ground truth (30.5m): Disqualification from the race results; if the race is suspended and cannot be re-started, 30 seconds are added to the driver's elapsed time instead.


,gold,chunk,bi_rank,bi_score,ce_rank,ce_score,text
0,,30.5 (Supply Of Tyres In The Champ),1,0.696,2,5.04,a set of tyres of the same specification. A penalty under Article 54.3d) will be imposed o…
1,,30.1 (Supply Of Tyres In The Champ),2,0.643,7,-0.74,least one (1) week before the start of the relevant Competition. This may only happen in C…
2,,30.5 (Supply Of Tyres In The Champ),3,0.636,3,2.56,"session or race is resumed in accordance with Article 58.10a), the use of wet- weather tyr…"
3,,30.5 (Supply Of Tyres In The Champ),4,0.611,5,0.22,returned no later than two (2) hours after the end of P3 j k) Once all tyres have been ret…
4,,30.5 (Supply Of Tyres In The Champ),5,0.608,4,1.93,30.5 Use of Tyres a) All tyres must be operated in accordance with the prescriptions issue…
5,,30.2 (Supply Of Tyres In The Champ),6,0.604,10,-2.04,"to each Competition for item iv) below, the FIA will provide all Competitors with the foll…"
6,,30.2 (Supply Of Tyres In The Champ),7,0.592,9,-1.40,of the tyres and any changes to the timing of electronic returns. • Procedures that will b…
7,,30.5 (Supply Of Tyres In The Champ),8,0.591,6,-0.63,the race. From the ten (10) remaining sets: iii) Two (2) sets must be electronically retur…
8,★,30.5 (Supply Of Tyres In The Champ),9,0.590,1,5.56,"or wet-weather tyres during the race, each driver must use at least two different specific…"
9,,30.5 (Supply Of Tyres In The Champ),10,0.585,8,-1.01,"by the FIA. Unless the additional tyres have not been made available, or the session is de…"


ANSWER 1 (baseline Top-3: ['30.5 (Supply Of Tyres In The Champ)', '30.1 (Supply Of Tyres In The Champ)', '30.5 (Supply Of Tyres In The Champ)'] )
According to Article 30.5, a penalty under Article 54.3d) will be imposed on any driver who does not change
tyres as specified. 

ANSWER 2 (re-ranked Top-3: ['30.5 (Supply Of Tyres In The Champ)', '30.5 (Supply Of Tyres In The Champ)', '30.5 (Supply Of Tyres In The Champ)'] )
Failure to use at least two different specifications of dry-weather tyres during a race will result in the
disqualification of the driver from the race results (Article 30.5). However, if the race is suspended and
cannot be re-started, a penalty of thirty (30) seconds will instead be added to the elapsed time of any driver
unable to comply with this requirement (Article 30.5).


**Observation:** The gold chunk is buried at #9 and the baseline Top-3 contains a *different* tyre penalty (54.3d for not changing tyres as specified). **Answer 1 misstates the rule**: it gives that unrelated 54.3d penalty as the penalty for the two-compound rule. After re-ranking (#9 → #1), **Answer 2** gives the correct sanction: disqualification, or +30 s if the race is suspended and cannot be re-started.

### Q21: Negation / constraint
**Query:** *Can mechanics work on the car while a driver serves a five second penalty?*

In [15]:
show_query("Q21")

[Q21] Negation / constraint
Q: Can mechanics work on the car while a driver serves a five second penalty?
Ground truth (54.4c): No. While stationary for a 5 s or 10 s time penalty the car may not be worked on until it has been stationary for the full penalty; touching the car or driver by hand, tools or equipment counts as work. A breach may lead to disqualification.


,gold,chunk,bi_rank,bi_score,ce_rank,ce_score,text
0,,54.4 (Incidents During The Sprint ),1,0.564,3,0.63,tools or equipment will all constitute working. d) Whilst a car is stationary in the pit l…
1,,58.5 (Resuming A Sprint Session Or),2,0.545,6,-2.14,"58.5 When the one (1) minute signal is shown, engines should be started and all team perso…"
2,,58.2 (Resuming A Sprint Session Or),3,0.502,5,-2.12,"58.2 Signals will be shown ten minutes, five minutes, three minutes, one minute and fiftee…"
3,★,54.4 (Incidents During The Sprint ),4,0.499,2,0.85,"he shall remain for the period of the time penalty. However, unless the driver was already…"
4,,37.5 (Practice Sessions),5,0.498,7,-4.74,"37.5 Any driver taking part in any free practice session, the qualifying session or the sp…"
5,,54.4 (Incidents During The Sprint ),6,0.492,10,-6.61,"54.4 Should the stewards decide to impose any of the penalties under Article 54.3a), 54.3b…"
6,,54.3 (Incidents During The Sprint ),7,0.485,1,1.91,54.3 The stewards may impose any one of the penalties below on any driver involved in an I…
7,,55.7 (Safety Car),8,0.466,8,-5.69,55.7 All competing cars must reduce speed and form up in line behind the safety car no mor…
8,,54.3 (Incidents During The Sprint ),9,0.461,9,-6.18,the driver’s next race. If any of the seven penalties above are imposed they shall not be …
9,,44.4 (Race Starting Procedure),10,0.459,4,-2.10,44.4 When the five (5) minute signal is shown all cars on the grid and any cars in the pit…


ANSWER 1 (baseline Top-3: ['54.4 (Incidents During The Sprint )', '58.5 (Resuming A Sprint Session Or)', '58.2 (Resuming A Sprint Session Or)'] )
No, while a car is stationary in the pit lane as a result of incurring a time penalty, it may not be worked on
(Article 54.4d). However, if the engine stops, any work necessary to restart it may be carried out after the
time penalty period has elapsed (Article 54.4d). Failing to comply with this requirement may result in the car
being disqualified (Article 54.4e). 

ANSWER 2 (re-ranked Top-3: ['54.3 (Incidents During The Sprint )', '54.4 (Incidents During The Sprint )', '54.4 (Incidents During The Sprint )'] )
No, mechanics cannot work on the car while the driver is serving a five-second time penalty. According to
Article 54.4c), a car that is stationary in the pit lane to serve a penalty incurred under Article 54.3a) may
not be worked on until it has been stationary for the duration of the penalty. Furthermore, Article 54.4c)
specifies that 

**Observation:** A negation query: the answer is the clause *may not be worked on until … stationary for the duration of the penalty* (54.4c), found at baseline rank #4. The baseline Top-3 holds 54.4d (the stop-and-go clause) plus two irrelevant restart-procedure chunks. **Answer 1** reaches the right yes/no answer but cites the wrong clause and adds the engine-restart allowance, which applies to the stop-and-go penalty, not to a 5 s penalty. **Answer 2** cites 54.4c and the definition of "working".

### Q26: Keyword trap / distractor
**Query:** *Can children enter the pit lane?*

In [16]:
show_query("Q26")

[Q26] Keyword trap / distractor
Q: Can children enter the pit lane?
Ground truth (26.13): Not at certain times: unless authorised by the FIA, no one under 16 may be in the pit lane from 15 min before to 5 min after each practice / qualifying / sprint-qualifying session, and from 15 min before the pit exit opens for reconnaissance laps until the last car enters parc fermé after the sprint or race.


,gold,chunk,bi_rank,bi_score,ce_rank,ce_score,text
0,,"34.3 (Pit Entry Road, Pit Lane And)",1,0.614,7,-2.11,34.3 At no time may a car be reversed in the pit lane under its own power. 34.4 The pit la…
1,,57.2 (Suspending A Sprint Session ),2,0.603,5,-1.11,"57.2 When the signal is given overtaking is forbidden, the pit exit will be closed and all…"
2,,55.11 (Safety Car),3,0.592,3,1.33,55.11 Under certain circumstances the clerk of the course may ask the cars and the safety …
3,,49.4 (Formation Lap Behind The Saf),4,0.585,1,1.59,49.4 Any other car entering the pit lane during the formation laps may re-join the track b…
4,★,26.13 (General Safety),5,0.568,2,1.38,26.13 Unless authorised by the FIA no one under the age of 16 is permitted in the pit lane…
5,,"34.15 (Pit Entry Road, Pit Lane And)",6,0.551,4,0.13,34.15 In exceptional circumstances the Race Director may ask for the pit entry to be close…
6,,58.5 (Resuming A Sprint Session Or),7,0.549,9,-2.67,who fails to start the race from the pit lane. If any driver needs assistance after the fi…
7,,"34.1 (Pit Entry Road, Pit Lane And)",8,0.547,8,-2.51,"34.1 Unless otherwise defined by the Race Director, the section of track leading to the pi…"
8,,55.2 (Safety Car),9,0.543,10,-3.88,"55.2 Prior to the pit lane opening for a reconnaissance lap, the safety car will leave the…"
9,,57.3 (Suspending A Sprint Session ),10,0.538,6,-1.91,57.3 Any cars unable to return to the pit lane as a result of the track being blocked will…


ANSWER 1 (baseline Top-3: ['34.3 (Pit Entry Road, Pit Lane And)', '57.2 (Suspending A Sprint Session )', '55.11 (Safety Car)'] )
The provided context does not answer this. 

ANSWER 2 (re-ranked Top-3: ['49.4 (Formation Lap Behind The Saf)', '26.13 (General Safety)', '55.11 (Safety Car)'] )
According to Article 26.13, no one under the age of 16 is permitted in the pit lane during specific time
periods surrounding practice, qualifying, sprint sessions, reconnaissance laps, and races, unless authorised
by the FIA. Outside of these restricted times, or with specific FIA authorization, individuals under 16 may
enter the pit lane.


**Observation:** A keyword trap: the query says *children*, the regulation says *no one under the age of 16*. The bi-encoder ranks generic pit-lane chunks ("car reversed in the pit lane", "pit exit will be closed") above the answer because they share *pit lane*. The cross-encoder links *children* with *under the age of 16* (#5 → #2). **Answer 1** says the context does not answer the question; **Answer 2** gives the age limit, the restricted time windows and the FIA-authorisation exception.

### Q31: Negation / constraint
**Query:** *When are no championship points awarded for a shortened race?*

In [17]:
show_query("Q31")

[Q31] Negation / constraint
Q: When are no championship points awarded for a shortened race?
Ground truth (6.5a-b): No points if the leader completed fewer than two laps; in the reduced-points cases, no points unless the leader completed at least two laps without Safety Car / VSC intervention.


,gold,chunk,bi_rank,bi_score,ce_rank,ce_score,text
0,,6.4 (World Championship),1,0.616,1,3.43,6.4 Points for both titles will be awarded at each Competition according to the following …
1,,7.1 (Dead Heat),2,0.608,4,-0.48,"7.1 Prizes and points awarded for all the positions of Competitors who tie, will be added …"
2,,6.5 (World Championship),3,0.605,5,-0.98,awarded in accordance with column 2 of the table below. e) If the leader has completed 50%…
3,★,6.5 (World Championship),4,0.601,2,2.56,6.5 If the race distance from the start signal to the end-of-session signal is less than t…
4,,5.5 (Championship Competitions),5,0.546,7,-3.04,5.5 The maximum number of Competitions in the Championship is 24 the minimum is 8. 5.6 The…
5,,6.6 (World Championship),6,0.530,3,0.12,6.6 If the sprint session distance from the start signal to the end-of-session signal is l…
6,,10.8 (Track Running Time Outside A),7,0.523,8,-4.93,and software of a specification that have been used in at least one (1) race or TCC during…
7,,23.6 (Operational Personnel During),8,0.519,9,-5.65,23.6 Each Competitor will be permitted the following exceptions during the Championship to…
8,,1.3 (Regulations),9,0.518,10,-7.57,"1.3 These Sporting Regulations apply to the whole calendar year referred to in the title, …"
9,,32.3 (Changes Of Driver),10,0.517,6,-2.89,32.3 Any new driver may score points in the Championship.…


ANSWER 1 (baseline Top-3: ['6.4 (World Championship)', '7.1 (Dead Heat)', '6.5 (World Championship)'] )
The provided context does not answer this. 

ANSWER 2 (re-ranked Top-3: ['6.4 (World Championship)', '6.5 (World Championship)', '6.6 (World Championship)'] )
For a shortened race, no points are awarded if the leader has completed less than two laps (Article 6.5a).
Additionally, no points are awarded unless a minimum of two laps have been completed by the leader without a
Safety Car and/or VSC intervention (Article 6.5b). Furthermore, no fastest lap point will be awarded if the
leader has completed less than 50% of the scheduled race distance (Article 6.4).


**Observation:** The question asks when **no** points are awarded. The bi-encoder ranks the *positive* points table (6.4) and the dead-heat rule (7.1) first, and the second half of 6.5 at #3, but that half only has the 25 %/50 %/75 % thresholds. The chunk with *No points will be awarded if the leader has completed less than two laps* moves from #4 to #2. **Answer 1** says the context does not answer the question; **Answer 2** gives both no-points conditions (fewer than 2 laps; fewer than 2 laps without SC/VSC).

### Q50: Buried chunk (rank 4-10)
**Query:** *What happens if a driver cannot serve a penalty because he retired?*

In [18]:
show_query("Q50")

[Q50] Buried chunk (rank 4-10)
Q: What happens if a driver cannot serve a penalty because he retired?
Ground truth (54.3): If a 5 s / 10 s / drive-through / stop-and-go penalty cannot be served because the driver retired, the stewards may impose a grid place penalty at the driver's next race.


,gold,chunk,bi_rank,bi_score,ce_rank,ce_score,text
0,,"34.14 (Pit Entry Road, Pit Lane And)",1,0.501,2,-0.26,"qualifying session, the stewards may drop the driver such number of grid positions as they…"
1,,54.3 (Incidents During The Sprint ),2,0.484,6,-3.16,the driver’s next race. If any of the seven penalties above are imposed they shall not be …
2,,49.5 (Formation Lap Behind The Saf),3,0.474,8,-6.14,49.5 A penalty in accordance with Article 54.3d) will be imposed on any driver whose tyre(…
3,,37.5 (Practice Sessions),4,0.472,5,-2.58,"37.5 Any driver taking part in any free practice session, the qualifying session or the sp…"
4,,26.5 (General Safety),5,0.459,10,-9.85,"26.5 A driver who abandons a car must leave it in neutral or with the clutch disengaged, w…"
5,★,54.3 (Incidents During The Sprint ),6,0.452,1,5.54,and then re-join the sprint session or the race. If any of the four (4) penalties above ar…
6,,32.5 (Changes Of Driver),7,0.433,7,-5.11,32.5 If one of the Competitor’s nominated drivers is unable to drive at some stage after t…
7,,58.12 (Resuming A Sprint Session Or),8,0.429,4,-2.22,A penalty in accordance with Article 54.3d) will be imposed on any driver who fails to res…
8,,54.4 (Incidents During The Sprint ),9,0.423,3,-0.37,"he shall remain for the period of the time penalty. However, unless the driver was already…"
9,,26.18 (General Safety),10,0.418,9,-8.75,"26.18 Except during a sprint session suspension or race suspension, any car abandoned on …"


ANSWER 1 (baseline Top-3: ['34.14 (Pit Entry Road, Pit Lane And)', '54.3 (Incidents During The Sprint )', '49.5 (Formation Lap Behind The Saf)'] )
The provided context does not answer this. 

ANSWER 2 (re-ranked Top-3: ['54.3 (Incidents During The Sprint )', '34.14 (Pit Entry Road, Pit Lane And)', '54.4 (Incidents During The Sprint )'] )
If a driver is unable to serve an imposed penalty due to retirement from a sprint session or race, the
stewards may impose a grid place penalty on the driver at his next race (Article 54.3). Additionally, if a
driver retires from a sprint session or race as a result of the car being released in an unsafe condition, a
fine may be imposed upon the Competitor (Article 34.14d).


**Observation:** An exception clause buried at #6 in the middle of the long 54.3 penalty list. The bi-encoder prefers chunks with *penalty / grid positions* (34.14, 49.5). The cross-encoder scores the clause *unable to serve the penalty due to retirement* far higher (+5.54 vs at most −0.26 for every other candidate) and moves it to #1. **Answer 1** says the context does not answer the question; **Answer 2** gives the grid-place penalty at the next race.

### Counter-example: when re-ranking hurts
The cross-encoder is not always right. In this query the gold chunk was already in the baseline Top-3 and re-ranking pushed it out.

### Q03: Keyword trap / distractor
**Query:** *Can the team top up fuel when the car comes in for new tyres?*

In [19]:
show_query("Q03")

[Q03] Keyword trap / distractor
Q: Can the team top up fuel when the car comes in for new tyres?
Ground truth (36.2): No. Fuel may not be added to or removed from a car after it has left the pit lane for the reconnaissance lap(s) until the end-of-session signal; refuelling is only allowed in the team's designated garage.


,gold,chunk,bi_rank,bi_score,ce_rank,ce_score,text
0,,30.5 (Supply Of Tyres In The Champ),1,0.524,2,-6.02,"by the FIA. Unless the additional tyres have not been made available, or the session is de…"
1,,30.5 (Supply Of Tyres In The Champ),2,0.501,4,-8.41,"session or race is resumed in accordance with Article 58.10a), the use of wet- weather tyr…"
2,★,36.1 (Refuelling),3,0.497,5,-8.46,36.1 Refuelling is only permitted in the Competitors’ designated garages. 36.2 Fuel may no…
3,,30.5 (Supply Of Tyres In The Champ),4,0.496,6,-8.48,30.5 Use of Tyres a) All tyres must be operated in accordance with the prescriptions issue…
4,,30.3 (Supply Of Tyres In The Champ),5,0.480,8,-9.19,one must present both tyres to the Technical Delegate. e) The use of tyres without appropr…
5,,30.5 (Supply Of Tyres In The Champ),6,0.479,7,-8.80,"or wet-weather tyres during the race, each driver must use at least two different specific…"
6,,36.3 (Refuelling),7,0.468,9,-9.32,"36.3 No car may be refuelled, nor may fuel be removed from a car, at a rate greater than 0…"
7,,30.5 (Supply Of Tyres In The Champ),8,0.468,1,-5.58,sensors may be fitted on the car to measure tyre state and or performance and must be agre…
8,,10.8 (Track Running Time Outside A),9,0.464,3,-7.65,elements used in the Competitions of the current year. e) A maximum of forty (40) car days…
9,,"34.9 (Pit Entry Road, Pit Lane And)",10,0.456,10,-10.72,"34.9 With the exception of the reconnaissance laps permitted by Articles 43.2 and 44.1, an…"


ANSWER 1 (baseline Top-3: ['30.5 (Supply Of Tyres In The Champ)', '30.5 (Supply Of Tyres In The Champ)', '36.1 (Refuelling)'] )
According to Article 36.1, refuelling is only permitted in the Competitors’ designated garages. Furthermore,
under Article 36.2, fuel may not be added to a car after it has left the pit lane to start reconnaissance
lap(s) until the end-of-session signal has been shown. While the provided context does not specifically
mention tyre changes, fuel cannot be added to a car outside the designated garage or after it leaves the pit
lane until the end of the session (Articles 36.1 and 36.2). 

ANSWER 2 (re-ranked Top-3: ['30.5 (Supply Of Tyres In The Champ)', '30.5 (Supply Of Tyres In The Champ)', '10.8 (Track Running Time Outside A)'] )
The provided context does not answer this.


**Observation:** The baseline had the refuelling chunk (36.1/36.2) at #3, and **Answer 1 is correct** (fuel may only be added in the garage and not between the reconnaissance lap and the end of the race). The query is phrased around *tyres*, so the cross-encoder fills the Top-3 with tyre and test-day chunks. Every cross-encoder score here is strongly negative (−5.6 to −10.7), meaning it finds no candidate convincing, and it pushes the refuelling chunk to #5. **Answer 2** says the context does not answer the question. MS MARCO re-rankers are trained on short web queries and can over-weight the dominant topic word in a mixed-topic question.

## 8. Latency profiling

Retrieval stages are timed over every query in the suite, repeated `N_REPEATS` times after warm-up. The LLM time is the
wall-clock time of the Gemini calls made in Section 7 (network round trip included; for cached answers this is the
latency recorded when the answer was first generated).

In [20]:
N_REPEATS = 20
lat = {"embed_ms": [], "search_ms": [], "rerank_ms": []}
for _ in range(N_REPEATS):
    for q in QUERIES:
        cands, t_b = baseline_retrieve(q["query"])
        _, t_r = rerank(q["query"], cands)
        lat["embed_ms"].append(t_b["embed_ms"]); lat["search_ms"].append(t_b["search_ms"])
        lat["rerank_ms"].append(t_r["rerank_ms"])

b_total = np.array(lat["embed_ms"]) + np.array(lat["search_ms"])
r_total = b_total + np.array(lat["rerank_ms"])
llm = np.array([t for t in llm_log if t == t])

def stats(name, arr):
    return {"stage": name, "mean_ms": arr.mean(), "p50_ms": np.percentile(arr, 50), "p95_ms": np.percentile(arr, 95)}

latency_df = pd.DataFrame([
    stats("Query embedding (bi-encoder)", np.array(lat["embed_ms"])),
    stats("FAISS Top-10 search", np.array(lat["search_ms"])),
    stats("Cross-encoder re-rank (10 pairs)", np.array(lat["rerank_ms"])),
    stats("Pipeline 1 retrieval total", b_total),
    stats("Pipeline 2 retrieval total", r_total),
    stats("LLM generation (Gemini, per call)", llm),
    stats("Pipeline 1 end-to-end (retrieval + LLM)", b_total.mean() + llm),
    stats("Pipeline 2 end-to-end (retrieval + LLM)", r_total.mean() + llm),
]).round(2)
print(f"device={DEVICE}, {len(lat['embed_ms'])} timed retrievals; re-ranking overhead = "
      f"{np.mean(lat['rerank_ms']):.1f} ms ({np.mean(r_total) / np.mean(b_total):.1f}× the baseline retrieval time)")
latency_df

device=mps, 1040 timed retrievals; re-ranking overhead = 44.8 ms (9.4× the baseline retrieval time)


,stage,mean_ms,p50_ms,p95_ms
0,Query embedding (bi-encoder),5.34,5.31,5.67
1,FAISS Top-10 search,0.02,0.02,0.03
2,Cross-encoder re-rank (10 pairs),44.82,45.29,50.23
3,Pipeline 1 retrieval total,5.37,5.34,5.70
4,Pipeline 2 retrieval total,50.18,50.66,55.65
5,"LLM generation (Gemini, per call)",6526.09,5274.84,12230.76
6,Pipeline 1 end-to-end (retrieval + LLM),6531.45,5280.21,12236.12
7,Pipeline 2 end-to-end (retrieval + LLM),6576.27,5325.03,12280.94


## 9. Scaling experiment: why not run the cross-encoder over the whole corpus?

A bi-encoder embeds documents **once, offline**, so a query costs one forward pass plus a vector search. A cross-encoder
has nothing to pre-compute, because every (query, chunk) pair needs its own full transformer pass at query time. Below
we score **all** chunks with the cross-encoder for one query and extrapolate linearly to larger collections.

In [21]:
query = QUERIES[0]["query"]
all_pairs = [(query, t) for t in texts]
_ = cross_encoder.predict(all_pairs[:64], batch_size=64)  # warm-up

t0 = time.perf_counter(); full_scores = cross_encoder.predict(all_pairs, batch_size=64); sync()
ce_full_ms = (time.perf_counter() - t0) * 1e3
t0 = time.perf_counter()
for _ in range(20): baseline_retrieve(query)
bi_ms = (time.perf_counter() - t0) * 1e3 / 20

per_pair_ms = ce_full_ms / len(texts)
embed_ms, search_ms = np.mean(lat["embed_ms"]), np.mean(lat["search_ms"])

def fmt(ms):
    if ms < 1e3: return f"{ms:,.1f} ms"
    if ms < 6e4: return f"{ms / 1e3:,.1f} s"
    return f"{ms / 6e4:,.1f} min" if ms < 3.6e6 else f"{ms / 3.6e6:,.1f} h"

# exact (flat) search grows linearly but is a cheap dot product; ANN indexes (HNSW/IVF) grow sub-linearly
scale = pd.DataFrame([{
    "collection size (chunks)": f"{n:,}",
    "bi-encoder: embed query + flat search": fmt(embed_ms + search_ms * n / len(texts)),
    "two-stage: + cross-encoder on Top-10": fmt(embed_ms + search_ms * n / len(texts) + np.mean(lat["rerank_ms"])),
    "cross-encoder over ALL chunks": fmt(per_pair_ms * n),
} for n in [len(texts), 10_000, 100_000, 1_000_000, 10_000_000]])
print(f"Cross-encoder over all {len(texts)} chunks: {ce_full_ms:.0f} ms ({per_pair_ms:.2f} ms/pair) "
      f"vs bi-encoder retrieval {bi_ms:.1f} ms")
top3_full = [label(chunks[i]) for i in np.argsort(-full_scores)[:3]]
print("Top-3 when the cross-encoder scores the full corpus:", top3_full)
scale

Cross-encoder over all 526 chunks: 3385 ms (6.43 ms/pair) vs bi-encoder retrieval 5.5 ms
Top-3 when the cross-encoder scores the full corpus: ['36.1 (Refuelling)', '57.5 (Suspending A Sprint Session )', '57.3 (Suspending A Sprint Session )']


,collection size (chunks),bi-encoder: embed query + flat search,two-stage: + cross-encoder on Top-10,cross-encoder over ALL chunks
0,526,5.4 ms,50.2 ms,3.4 s
1,"10,000",5.8 ms,50.6 ms,1.1 min
2,"100,000",9.9 ms,54.7 ms,10.7 min
3,"1,000,000",50.7 ms,95.5 ms,1.8 h
4,"10,000,000",458.8 ms,503.6 ms,17.9 h


In [22]:
# Save all measured results (used to write the report)
Path("results").mkdir(exist_ok=True)
rank_df.to_csv("results/rank_table.csv", index=False)
latency_df.to_csv("results/latency.csv", index=False)
scale.to_csv("results/scaling.csv", index=False)
json.dump({"summary": summary.to_dict(orient="records"), "per_category": per_cat.reset_index().to_dict(orient="records"),
           "answers": [{k: q.get(k) for k in ("id", "query", "truth", "answer_1", "answer_2")} for q in QUERIES if "answer_1" in q],
           "ce_full_ms": ce_full_ms, "per_pair_ms": per_pair_ms, "index_time_s": index_time, "n_chunks": len(chunks),
           "device": DEVICE, "top3_full_corpus": top3_full},
          open("results/run_results.json", "w"), indent=1, default=str)
print("saved results/")

saved results/


## 10. Summary

* Section 6 gives the aggregate effect: how often the gold chunk reaches the LLM's Top-3 context with and without re-ranking.
* Section 7 shows, query by query, how each rank shift changed the generated answer.
* Section 8 shows the price: re-ranking 10 candidates adds tens of milliseconds, small next to the LLM call.
* Section 9 shows why the cross-encoder is used only as a second stage: its cost grows linearly with the number of
  chunks and cannot be pre-computed, so it is affordable for 10 candidates but not for a whole collection.

The written analysis is in `report/report.pdf`.